# linear model with Regularization

In [1]:
import os

# Install the specified wheel file using pip
os.system("pip install rtdl_num_embeddings-0.0.11-py3-none-any.whl")

# Rename the file 'onebap.bin' to 'onebap.so'
os.system("mv onebap.bin onebap.so")

# Import necessary libraries
import pickle  # For serializing and deserializing Python objects
import torch  # PyTorch for deep learning
import torch.nn as nn  # Neural network modules
import torch.nn.functional as F  # Functions for neural networks
import torch.optim  # Optimizers for training
from torch.utils.data import Dataset, DataLoader, TensorDataset  # Data handling utilities
from sklearn.model_selection import train_test_split  # Splitting data into training and testing sets

In [2]:
from sklearn.metrics import r2_score  # For calculating R-squared score
import pandas as pd  # Data manipulation and analysis
import math  # Mathematical functions
import numpy as np  # Numerical computations
from tqdm import tqdm  # Progress bar for loops
import polars as pl  # Alternative to pandas for dataframes
from collections import OrderedDict  # Ordered dictionary for maintaining insertion order
import sys  # System-specific parameters and functions
# from tabm_reference import Model, make_parameter_groups  # Custom model and utility functions
import warnings  # For suppressing warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

In [3]:

import joblib  # For saving and loading Python objects
import gc
import warnings

In [39]:
# Read the Parquet file
data = pl.read_parquet('processed_data/Processed_data_1695.parquet')

# Add a 'row_id' column, starting from 0 and sequentially incrementing
data = data.with_row_count("row_id", offset=0)

# Add lagged columns for each responder (responder_{idx}_lag_1)
for idx in range(9):
    lag_col_name = f"responder_{idx}_lag_1"  # Name of the lagged column
    original_col_name = f"responder_{idx}"  # Name of the original column
    # Create the lagged column by shifting the original column by 1 row
    data = data.with_columns(pl.col(original_col_name).shift(1).alias(lag_col_name))

# Print the data after adding 'row_id' and lagged columns
print("\nData after adding 'row_id' and lagged columns:")


Data after adding 'row_id' and lagged columns:


In [40]:
data

row_id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,…,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_0,responder_1,responder_2,responder_3,responder_4,responder_5,responder_6,responder_7,responder_8,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
u32,i16,i16,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,1695,0,0,3.373552,2.776059,1.035769,1.790385,1.915069,1.765592,-0.105391,-0.16963,-0.345125,0.03763,11.0,7.0,76.0,-1.029333,0.271748,-0.640703,-0.688929,-0.763758,-0.664928,-1.701001,-1.246213,1.171051,-0.151066,1.780195,0.592178,2.011824,1.034978,1.035668,1.118305,0.616451,-0.932135,-1.078001,-0.154166,…,-0.019777,1.419774,-0.401114,-0.292407,-0.196831,-1.569348,-2.423477,-0.725855,0.203034,-0.451438,-0.841593,0.31118,-0.575253,0.171726,0.140167,1.225793,0.929635,0.067034,0.06966,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505,null,null,null,null,null,null,null,null,null
1,1695,0,1,2.802384,1.901147,1.029203,2.146977,2.811388,1.423654,-0.105936,-0.173366,-0.382062,0.045153,11.0,7.0,76.0,-1.263074,0.253009,-0.599086,-0.688929,-0.633995,-0.664928,-1.906501,-0.91079,0.886705,0.012082,1.116906,0.85608,1.12455,0.723127,-1.920028,-0.053758,1.010441,-0.559132,-0.940507,0.010087,…,-0.079671,1.419774,-0.49611,-0.439725,-0.442559,-1.185965,-1.933517,-1.124566,0.131424,-0.604832,-0.760508,0.142326,-0.745543,0.171726,0.140167,-0.226083,-0.213198,-0.206452,-0.373929,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505
2,1695,0,2,2.506616,2.801315,1.376992,2.313733,2.135918,1.765473,-0.173767,-0.261385,-0.473618,0.018804,81.0,2.0,59.0,-0.99006,0.873093,-0.564999,-0.688929,-0.389982,-0.664928,-1.932154,-0.971507,0.023837,-0.145004,0.877403,0.370831,0.23665,-0.146222,0.580706,0.660125,0.125094,-0.627553,-0.784622,-0.183313,…,1.418136,1.419774,-0.363737,-0.314952,-0.355761,-1.836319,-1.781683,-0.99552,-0.030046,-0.476795,-0.888185,1.50716,-0.264173,0.171726,0.140167,1.517169,1.503251,0.094979,0.109544,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547
3,1695,0,3,1.868808,2.492323,1.293729,2.051276,2.601039,2.115296,-0.137015,-0.15458,-0.39843,0.058264,4.0,3.0,11.0,-1.047425,1.116742,-0.303803,-0.688929,-0.510801,-0.664928,-1.360154,-1.818448,-0.282608,-0.068771,0.591393,0.9088,0.051669,-0.261807,-0.01116,-0.169174,-0.219266,-0.642355,-0.530943,-0.037706,…,-0.0473,1.419774,-0.315971,-0.243883,-0.276656,-2.015491,-2.266636,-1.011813,0.711239,-0.346282,-1.06627,0.20662,-0.518951,0.171726,0.140167,0.425632,0.424689,-0.129259,-0.146268,0.047888,-0.01461,-0.334644,-0.0426,-0.569183,-0.244618,-0.055909,-0.570931,-0.120065,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755
4,1695,0,4,2.826087,2.754668,0.981692,2.405406,2.504181,2.060546,-0.067486,-0.153702,-0.205151,0.021181,15.0,1.0,9.0,-0.840723,1.097859,-0.419687,-0.688929,-0.834465,-0.664928,-1.323989,-1.13767,-1.158483,0.721674,0.600382,0.

In [41]:
weights = data['weight'].to_numpy().tolist()
train = data.drop(['weight'])

del data
gc.collect()

1561

### Get X,y 

In [42]:
cols=[f'feature_0{i}' if i<10 else f'feature_{i}' for i in range(79)]
X=train.select(cols).fill_null(3).to_numpy()
y=train.select('responder_6').to_numpy().flatten()
del train
gc.collect()

0

## Train test split

In [43]:
split_ratio = 0.2  # 20% for test data, 80% for training  
split = int(len(X) * split_ratio)  # Calculate the split index dynamically 
train_X,train_y,test_X,test_y,train_weight,test_weight=X[:-split],y[:-split],X[-split:],y[-split:],weights[:-split],weights[-split:]
print(f"train_X.shape:{train_X.shape},test_X.shape:{test_X.shape}")

train_X.shape:(120032, 79),test_X.shape:(30008, 79)


## Ridge Model

Ridge Regression is a type of linear regression that introduces a regularization term to address the problem of multicollinearity and overfitting in machine learning models. It is particularly useful when the predictors in the dataset are highly correlated.

The Ridge Regression model minimizes the following loss function:


$$
\text{Loss} = \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^{p} \beta_j^2
$$

Where:  
- $ y_i $: Observed dependent variable for the $ i $-th sample.  
- $ \beta_0 $: Intercept term of the model.  
- $ \beta_j $: Coefficients associated with the $ j $-th predictor $ x_{ij} $.  
- $ \lambda $: Regularization parameter that penalizes large coefficients.  
- $ n $: Total number of samples (observations).  
- $ p $: Total number of predictors (features).  

The first term represents the residual sum of squares (RSS), measuring the error between the observed and predicted values. The second term, the regularization term, is the penalty proportional to the sum of squared coefficients $ \beta_j $. The regularization parameter $ \lambda $ determines the magnitude of this penalty. A higher value of $ \lambda $ shrinks coefficients more, reducing model complexity.

When $ \lambda = 0 $, Ridge Regression is equivalent to ordinary least squares (OLS) regression. As $ \lambda $ increases, it imposes a higher penalty on the size of coefficients, effectively shrinking them towards zero but never making them exactly zero. This differs from Lasso regression, where coefficients can become exactly zero, effectively performing feature selection.

Ridge Regression is particularly advantageous when the dataset contains many highly correlated predictors, as it stabilizes the estimation of coefficients by reducing their variance. However, it does not perform variable selection, as all features are retained in the final model, albeit with smaller coefficients.

The regularization parameter $ \lambda $ is typically chosen through techniques like cross-validation to strike a balance between bias and variance, optimizing predictive performance.

In [9]:
from sklearn.linear_model import Ridge

### Fit 

In [10]:
def custom_metric(y_true,y_pred,weight):
    weighted_r2=1-(np.sum(weight*(y_true-y_pred)**2)/np.sum(weight*y_true**2))
    return weighted_r2


$$
\text{Weighted } R^2 = 1 - \frac{\sum_{i=1}^{n} w_i (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} w_i y_i^2}
$$

where $y_i$ is the actual value of the sample $i$, $hat{y}_i$ is the predicted value of the sample $i$, $w_i$ is the weight of the sample of the first $i$, and $n$ is the total number of samples.


In [44]:
model=Ridge()
model.fit(train_X,train_y)
train_pred=model.predict(train_X)
test_pred=model.predict(test_X)
print(f"train weighted_r2:{custom_metric(train_y,train_pred,weight=train_weight)}")
print(f"test weighted_r2:{custom_metric(test_y,test_pred,weight=test_weight)}")

train weighted_r2:0.019854180798005272
test weighted_r2:-0.03333866822942233


## Lasso

Lasso Regression is a type of linear regression that incorporates an L1 regularization term to enhance model performance by preventing overfitting and enabling feature selection. Unlike traditional linear regression, Lasso Regression can shrink some coefficients exactly to zero, effectively selecting a simpler model that only includes the most significant predictors.

The Lasso Regression model minimizes the following loss function:

$$
\text{Loss} = \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^{p} |\beta_j|
$$

Where:
- $ y_i $ is the observed dependent variable for the $ i $-th sample.
- $ \beta_0 $ is the intercept term of the model.
- $ \beta_j $ are the coefficients for the features $ x_{ij} $.
- $ \lambda $ (lambda) is the regularization parameter that controls the amount of shrinkage applied to the coefficients.
- $ n $ is the number of samples, and $ p $ is the number of features.

The first term in the loss function represents the residual sum of squares (RSS), which measures the discrepancy between the observed values and the values predicted by the model. The second term is the L1 regularization penalty, which is the sum of the absolute values of the coefficients $ \beta_j $. This penalty encourages sparsity in the model coefficients, meaning that it can drive some coefficients to exactly zero when $ \lambda $ is sufficiently large. This property makes Lasso Regression particularly useful for feature selection, as it effectively removes less important features from the model, leading to a more interpretable and potentially more generalizable model.

The regularization parameter $ \lambda $ plays a crucial role in balancing the trade-off between fitting the training data well and keeping the model coefficients small to enhance generalization. When $ \lambda = 0 $, Lasso Regression reduces to ordinary least squares (OLS) regression without any regularization. As $ \lambda $ increases, the penalty for larger coefficients becomes more significant, promoting simplicity in the model by retaining only the most impactful features.

Lasso Regression is especially advantageous in scenarios where there are many predictors, some of which may be irrelevant or highly correlated. By performing both regularization and feature selection, Lasso helps in building models that are easier to interpret and less prone to overfitting, thereby improving predictive performance on unseen data.

In [36]:
from sklearn.linear_model import Lasso  

### Fit 

In [45]:
model = Lasso(alpha=1.0)  
model.fit(train_X, train_y)  

train_pred = model.predict(train_X)  
test_pred = model.predict(test_X)  

print(f"train weighted_r2:{custom_metric(train_y,train_pred,weight=train_weight)}")
print(f"test weighted_r2:{custom_metric(test_y,test_pred,weight=test_weight)}")

train weighted_r2:0.00021439690239011266
test weighted_r2:-0.00486062813733068


## Elastic Net

Elastic Net Regression is a sophisticated linear regression technique that combines the penalties of both Lasso (L1) and Ridge (L2) regression methods. This hybrid approach leverages the strengths of both regularization techniques to enhance model performance, particularly in scenarios where there are multiple correlated predictors. Elastic Net is especially effective when the number of predictors exceeds the number of observations or when there are groups of highly correlated variables.

The Elastic Net model minimizes the following loss function:

$$
\text{Loss} = \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda_1 \sum_{j=1}^{p} |\beta_j| + \lambda_2 \sum_{j=1}^{p} \beta_j^2
$$

Here, $ y_i $ represents the observed dependent variable for the $ i $-th sample, $ \beta_0 $ is the intercept term, and $ \beta_j $ are the coefficients for the $ j $-th feature $ x_{ij} $. The parameters $ \lambda_1 $ and $ \lambda_2 $ are regularization parameters that control the influence of the L1 and L2 penalties, respectively. The term $ n $ denotes the number of samples, and $ p $ is the number of features.

The first part of the loss function, $ \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 $, represents the residual sum of squares (RSS), which measures the discrepancy between the observed values and those predicted by the model. The second term, $ \lambda_1 \sum_{j=1}^{p} |\beta_j| $, is the L1 regularization penalty introduced by Lasso regression, which encourages sparsity in the model coefficients by driving some of them to exactly zero. The third term, $ \lambda_2 \sum_{j=1}^{p} \beta_j^2 $, is the L2 regularization penalty from Ridge regression, which discourages large coefficients by penalizing their squared magnitudes.

By combining these two penalties, Elastic Net addresses some of the limitations inherent in using Lasso or Ridge individually. The L1 penalty promotes feature selection by eliminating irrelevant predictors, while the L2 penalty maintains the grouping effect, ensuring that strongly correlated predictors are either all included or excluded together. This makes Elastic Net particularly useful in high-dimensional datasets where predictors are highly correlated or when there are more predictors than observations.

The regularization parameters $ \lambda_1 $ and $ \lambda_2 $ play a crucial role in balancing the trade-off between bias and variance. Tuning these parameters, typically through cross-validation, allows the model to achieve optimal predictive performance by controlling the extent of regularization applied to the coefficients. When $ \lambda_1 = 0 $ and $ \lambda_2 > 0 $, Elastic Net reduces to Ridge Regression. Conversely, when $ \lambda_2 = 0 $ and $ \lambda_1 > 0 $, it becomes equivalent to Lasso Regression. When both $ \lambda_1 $ and $ \lambda_2 $ are greater than zero, Elastic Net benefits from the combined regularization, offering a more flexible approach to model fitting.

Elastic Net Regression is particularly advantageous in situations where there are multiple correlated variables, as it tends to select groups of related features rather than selecting one feature from a group and ignoring the others. This leads to models that are both interpretable and robust, capable of handling complex datasets with intertwined predictor relationships. Additionally, Elastic Net can perform well even when the number of predictors exceeds the number of observations, making it a versatile tool in the arsenal of regression techniques for various machine learning applications.

In [14]:
from sklearn.linear_model import ElasticNet  

### Fit 

In [15]:

model = ElasticNet(alpha=1.0, l1_ratio=0.2)  
model.fit(train_X, train_y)  

train_pred = model.predict(train_X)  
test_pred = model.predict(test_X)  

print(f"train weighted_r2:{custom_metric(train_y,train_pred,weight=train_weight)}")
print(f"test weighted_r2:{custom_metric(test_y,test_pred,weight=test_weight)}")

train weighted_r2:0.0005135252066779117
test weighted_r2:-0.005323082407179713


## Why Ridge gets highest $Weight R^2$

In real noisy data, Ridge's L2 regularization smooths the model more effectively and reduces the impact of noise. Lasso and Elastic Net may not perform as consistently as Ridge in noisy situations.